# Notebook 3 — Agent-Based Trophoblast Population Model
### AMPK Signaling Case Study: Metformin at the Triad

Section 5 of the case study describes a biphasic, non-monotonic dose–response for the placental
context — a single "optimal" AMPK activation level that restores balance, with harm on either side.
That single-curve picture is a simplification: real placental tissue is a **heterogeneous
population** of trophoblast cells with varying baseline severity, drug perfusion, and individual
sensitivity.

This notebook builds an **agent-based model (ABM)**: each trophoblast is represented as an
individual "agent" with its own parameters, drawn from realistic distributions. Applying a uniform
metformin dose to the population reveals that the "therapeutic window" is not a sharp point but a
**distribution of outcomes** — some cells land safely inside it, others don't, even at the
population-optimal dose. This is closer to how a real, spatially and biologically variable placenta
would respond.

A short spatial extension (Part 3) adds simple nearest-neighbor influence between cells, turning
this into a genuine spatial agent-based model rather than just independent sampling.


In [ ]:
!pip install ipywidgets -q
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, FloatSlider


## Part 1 — Building a heterogeneous cell population

In [ ]:
def sflt1_individual(x, peak_shift, depth_scale, overshoot_scale):
    """
    Per-cell biphasic sFlt-1 response to added AMPK activation `x` (0-100).
    peak_shift      : shifts where this cell's optimum sits (heterogeneity in disease severity)
    depth_scale     : how strongly this cell benefits at its optimum
    overshoot_scale : how badly this cell is harmed by over-activation
    """
    x = np.clip(x, 0, 100)
    base = 90 - 60*depth_scale*np.sin(np.pi*np.clip(x + peak_shift, 0, 100)/100)
    overshoot = 25*overshoot_scale*(x/100)**4
    return base + overshoot


def make_population(n_cells, severity_mean=55, severity_sd=15, seed=None):
    """
    Each row = one trophoblast agent.
    baseline_ampk    : this cell's untreated AMPK state (disease severity proxy)
    peak_shift       : individual variation in where the therapeutic optimum falls
    depth_scale      : individual variation in benefit magnitude
    overshoot_scale  : individual variation in harm magnitude at high dose
    perfusion_noise  : local variation in effective drug delivery
    """
    rng = np.random.default_rng(seed)
    return {
        'baseline_ampk':   np.clip(rng.normal(severity_mean, severity_sd, n_cells), 0, 100),
        'peak_shift':      rng.normal(0, 8, n_cells),
        'depth_scale':     np.clip(rng.normal(1.0, 0.15, n_cells), 0.5, 1.5),
        'overshoot_scale': np.clip(rng.normal(1.0, 0.25, n_cells), 0.3, 2.0),
        'perfusion_noise': rng.normal(0, 8, n_cells),
    }


def apply_dose(pop, dose):
    """Apply a uniform prescribed dose; each cell's *effective* exposure varies with local perfusion."""
    effective_x = np.clip(dose + pop['perfusion_noise'], 0, 100)
    return sflt1_individual(effective_x, pop['peak_shift'], pop['depth_scale'], pop['overshoot_scale'])


### Population response across a dose range

In [ ]:
pop = make_population(n_cells=600, seed=3)
doses = [0, 25, 50, 75, 100]
THRESHOLD = 45  # sFlt-1 level below which a cell is considered "in the therapeutic window"

fig, axes = plt.subplots(1, len(doses), figsize=(16,3.2), sharey=True)
fractions = []
for ax, d in zip(axes, doses):
    sflt1 = apply_dose(pop, d)
    frac = np.mean(sflt1 < THRESHOLD)
    fractions.append(frac)
    ax.hist(sflt1, bins=22, range=(0,140), color='#B15A2E', alpha=0.8)
    ax.axvline(THRESHOLD, color='k', linestyle='--', linewidth=1)
    ax.set_title(f"dose={d}\n{frac:.0%} in window")
fig.suptitle('Population sFlt-1 distribution across dose levels', y=1.05)
plt.tight_layout(); plt.show()

print("Fraction of cells in therapeutic window at each dose:")
for d, f in zip(doses, fractions):
    print(f"  dose={d:3d} -> {f:.1%}")


### Population-level optimal dose curve
Sweep dose finely and plot the fraction of cells inside the therapeutic window — this is the population-level analog of the single-cell biphasic curve, and it should be *broader and flatter* due to heterogeneity.

In [ ]:
dose_range = np.linspace(0, 100, 41)
frac_curve = [np.mean(apply_dose(pop, d) < THRESHOLD) for d in dose_range]

plt.figure(figsize=(7,4.5))
plt.plot(dose_range, frac_curve, color='#B15A2E', linewidth=2)
best_dose = dose_range[np.argmax(frac_curve)]
plt.axvline(best_dose, color='k', linestyle=':', label=f'Population optimum \u2248 {best_dose:.0f}')
plt.xlabel('Prescribed dose (AMPK activation intensity)')
plt.ylabel('Fraction of cells in therapeutic window')
plt.title('Population-level dose-response is a distribution, not a point')
plt.legend(); plt.tight_layout(); plt.show()


## Part 2 — Interactive explorer
Vary population heterogeneity and see how a "sharper" vs "noisier" population changes how forgiving
the optimal dose is.

In [ ]:
@interact(
    n_cells=IntSlider(min=100, max=1500, step=100, value=600, description='Population'),
    severity_sd=FloatSlider(min=2, max=30, step=1, value=15, description='Severity SD'),
    dose=IntSlider(min=0, max=100, step=5, value=50, description='Dose'),
)
def explore(n_cells=600, severity_sd=15, dose=50):
    pop = make_population(n_cells, severity_sd=severity_sd, seed=1)
    sflt1 = apply_dose(pop, dose)
    frac = np.mean(sflt1 < THRESHOLD)

    fig, ax = plt.subplots(figsize=(7,4.2))
    ax.hist(sflt1, bins=25, range=(0,140), color='#B15A2E', alpha=0.8)
    ax.axvline(THRESHOLD, color='k', linestyle='--')
    ax.set_title(f"dose={dose}, severity SD={severity_sd}  ->  {frac:.0%} of cells in window")
    ax.set_xlabel('sFlt-1 level'); ax.set_ylabel('Number of cells')
    plt.tight_layout(); plt.show()


## Part 3 — Spatial extension (optional, bonus complexity)

Real placental tissue has local paracrine signaling: a stressed region can influence its
neighbors. This adds a simple 2D grid of cells with nearest-neighbor coupling, turning the model
into a genuine spatial agent-based simulation.

In [ ]:
def make_grid_population(size=25, severity_mean=55, severity_sd=15, seed=None):
    rng = np.random.default_rng(seed)
    baseline = np.clip(rng.normal(severity_mean, severity_sd, (size,size)), 0, 100)
    peak_shift = rng.normal(0, 8, (size,size))
    depth_scale = np.clip(rng.normal(1.0, 0.15, (size,size)), 0.5, 1.5)
    overshoot_scale = np.clip(rng.normal(1.0, 0.25, (size,size)), 0.3, 2.0)
    perfusion_noise = rng.normal(0, 8, (size,size))
    return dict(baseline=baseline, peak_shift=peak_shift, depth_scale=depth_scale,
                overshoot_scale=overshoot_scale, perfusion_noise=perfusion_noise, size=size)


def grid_sflt1(grid, dose, coupling=0.15, n_steps=5):
    """
    Compute each cell's sFlt-1 given the dose, then let neighboring cells' stress
    partially influence each other over a few local-diffusion steps (simple paracrine effect).
    """
    size = grid['size']
    effective_x = np.clip(dose + grid['perfusion_noise'], 0, 100)
    sflt1 = sflt1_individual(effective_x, grid['peak_shift'], grid['depth_scale'], grid['overshoot_scale'])

    for _ in range(n_steps):
        neighbor_avg = (
            np.roll(sflt1, 1, axis=0) + np.roll(sflt1, -1, axis=0) +
            np.roll(sflt1, 1, axis=1) + np.roll(sflt1, -1, axis=1)
        ) / 4
        sflt1 = (1-coupling)*sflt1 + coupling*neighbor_avg
    return sflt1


grid = make_grid_population(size=25, seed=4)
fig, axes = plt.subplots(1, 3, figsize=(13,4))
for ax, d in zip(axes, [0, 50, 100]):
    field = grid_sflt1(grid, d)
    im = ax.imshow(field, cmap='RdYlGn_r', vmin=0, vmax=120)
    ax.set_title(f'dose={d}'); ax.axis('off')
fig.colorbar(im, ax=axes, shrink=0.7, label='sFlt-1 level')
plt.suptitle('Spatial trophoblast field: paracrine smoothing of the dose response', y=1.03)
plt.show()


### Discussion prompts for your write-up

- The population optimal dose curve (Part 1) is broader/flatter than any single cell's curve.
  What does that imply about choosing a single clinical dose for a genetically and physiologically
  diverse patient population?
- In Part 3, does increasing `coupling` make the tissue *more* or *less* forgiving of an
  off-target dose? What biological process might `coupling` represent (hint: paracrine cytokine or
  ROS signaling between adjacent cells)?
- How would you extend this model to represent *disease severity* varying not just cell-to-cell,
  but *patient-to-patient* (i.e., add a second level of hierarchy)?
